# Streaming

<img src="../assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.3     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.5     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-community    | >=0.4.2  | 0.4.2     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/pyth

In [3]:
from langchain.agents import create_agent

In [4]:
agent = create_agent(
    model="openai:gpt-5",
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [5]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Why did the full-stack developer break up with the database? Every time they tried to commit, it rolled back.


## values
You have seen this streaming mode in our examples so far. 

In [6]:
# Stream = values
for step in agent.stream({"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
                         stream_mode="values"):
    step["messages"][-1].pretty_print()  # print the last message in the stream

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

I only know 25 letters of the alphabet. I don’t know y.


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [7]:
for token, metadata in agent.stream (
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="", flush=True)

Breakfast is a bustling town square,
spoons are tiny drummers,
orange juice is sunshine in a glass,
and there’s always a polite negotiation
about the last pancake.

Mom flips them like small moons.
Dad flips a pun.
We groan on purpose,
then grin when it boomerangs back as a bigger laugh.

The dog is Chief Crumb Inspector,
snout to the floor,
tail ticking like a happy clock
that only knows snack o’clock.

Socks go missing on epic quests,
reuniting in unlikely partnerships.
We call the laundry basket Narnia,
because time behaves differently there.

We race the rain to the porch,
count thunder like hopscotch,
and clap for the sunset
as if it just finished a school play.

Grandma plants stories in our ears;
they sprout into bedtime jungles.
Grandpa waters them with chuckles
till the living room grows leaves.

We stack our worries by the door like umbrellas
and leave them for tomorrow’s weather.
Then we dance—careful around the lamp—
and pass the remote like a relay baton.

When nightlights

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [10]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"The current weather in {city} is sunny with a temperature of 25°C."

agent = create_agent(model="openai:gpt-5-mini", tools=[get_weather]);

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in New York?"}]},
    stream_mode=["values", "custom"],
):print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='29bbf611-e6ae-4b4f-9f84-1ab7e7dbc7fd')]})
('values', {'messages': [HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='29bbf611-e6ae-4b4f-9f84-1ab7e7dbc7fd'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 133, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYHTpbm91YKifYZVCNrwtKkENWWje', 'service_tier': 'default', 'finish_reason': 'tool_call